In [1]:
import pandas as pd
df = pd.read_csv("/kaggle/input/quickenreviews/quicken_reviews.csv")
df

,rating,content
0,5,I began using Quicken in the DOS version. It ...
1,5,answered all my questions and were friendly
2,1,it is impossible for me to update my current v...
3,5,"easy to use, organize, and store years of data"
4,1,Forced subscription is wrong. Let me just buy ...
...,...,...
44733,3,should not have to renew subscription when I d...
44734,2,Online version is too limited. UI is awkward ...
44735,1,I just want to use Quicken on one device my de...
44736,1,Used to be one of the best products out there....


In [ ]:
df = df.dropna(subset=['rating', 'content'])

df = df[
    (df['rating'] != 3) & # rating 3 represente les avis neutres qui nous intéresse pas
    (df['rating'] != 4) & 
    (df['content'].str.strip() != '')
].copy()

df.reset_index(drop=True, inplace=True)

df

,rating,content
0,5,I began using Quicken in the DOS version. It ...
1,5,answered all my questions and were friendly
2,1,it is impossible for me to update my current v...
3,5,"easy to use, organize, and store years of data"
4,1,Forced subscription is wrong. Let me just buy ...
...,...,...
25872,5,Excellent help. Competent service rep.
25873,2,Online version is too limited. UI is awkward ...
25874,1,I just want to use Quicken on one device my de...
25875,1,Used to be one of the best products out there....


In [3]:
df['label'] = df['rating'].apply(lambda x: 0 if x <= 2 else  1)

In [4]:
df = df.rename(columns={'content': 'text'})
df

,rating,text,label
0,5,I began using Quicken in the DOS version. It ...,1
1,5,answered all my questions and were friendly,1
2,1,it is impossible for me to update my current v...,0
3,5,"easy to use, organize, and store years of data",1
4,1,Forced subscription is wrong. Let me just buy ...,0
...,...,...,...
25872,5,Excellent help. Competent service rep.,1
25873,2,Online version is too limited. UI is awkward ...,0
25874,1,I just want to use Quicken on one device my de...,0
25875,1,Used to be one of the best products out there....,0


In [5]:
df['label'].value_counts()

label
1    16344
0     9533
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [8]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

2025-04-18 11:51:53.368836: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744977113.600725      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744977113.667994      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [9]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/20701 [00:00<?, ? examples/s]

Map:   0%|          | 0/5176 [00:00<?, ? examples/s]

In [10]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
training_args = TrainingArguments(
    output_dir="./ReputationAnalyzer_BERT",
    eval_strategy="steps", 
    eval_steps=200,
    save_strategy="best",  
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    report_to=[],  
)

In [34]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [37]:
import os 

os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,F1
200,No log,0.098723,0.973725,0.979160
400,No log,0.112258,0.968895,0.974926
600,0.060700,0.094256,0.971793,0.977287
800,0.060700,0.106013,0.976816,0.981533
1000,0.050400,0.095156,0.978168,0.982623
1200,0.050400,0.093614,0.978748,0.983103
1400,0.050400,0.101139,0.977975,0.982499
1600,0.023200,0.110888,0.978362,0.982769
1800,0.023200,0.110485,0.979328,0.983561


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=1941, training_loss=0.03739106513863062, metrics={'train_runtime': 1470.7322, 'train_samples_per_second': 42.226, 'train_steps_per_second': 1.32, 'total_flos': 5296541169372300.0, 'train_loss': 0.03739106513863062, 'epoch': 3.0})

In [39]:
best_model = BertForSequenceClassification.from_pretrained("/kaggle/working/ReputationAnalyzer_BERT/checkpoint-1800")

In [48]:
from sklearn.metrics import classification_report

trainer_b = Trainer(
    model=best_model,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

predictions = trainer_b.predict(test_dataset)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

print(classification_report(y_true, y_pred, digits=4))

/tmp/ipykernel_31/1248133263.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_b = Trainer(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9649    0.9795    0.9722      1907
           1     0.9880    0.9792    0.9836      3269

    accuracy                         0.9793      5176
   macro avg     0.9764    0.9794    0.9779      5176
weighted avg     0.9795    0.9793    0.9794      5176



In [53]:
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)
best_model.eval()

negative_text = "I'm extremely disappointed. The package arrived broken and no one is replying to my complaints."
positive_text = "I use this app every day and absolutely love it. Highly recommended!"

inputs1 = tokenizer(negative_text, return_tensors="pt", truncation=True, padding=True)
inputs1 = {key: val.to(device) for key, val in inputs1.items()}

with torch.no_grad():
    outputs1 = best_model(**inputs1)
    logits1 = outputs1.logits
    predicted_class1 = torch.argmax(logits1, dim=1).item()

inputs2 = tokenizer(positive_text, return_tensors="pt", truncation=True, padding=True)
inputs2 = {key: val.to(device) for key, val in inputs2.items()}

with torch.no_grad():
    outputs2 = best_model(**inputs2)
    logits2 = outputs2.logits
    predicted_class2 = torch.argmax(logits2, dim=1).item()

label_map = {0: "Negative", 1: "Positive"}
print(f"Predicted sentiment: {label_map[predicted_class1]}")
print(f"Predicted sentiment: {label_map[predicted_class2]}")

Predicted sentiment: Negative
Predicted sentiment: Positive
